In [1]:
# Task 2 (Section C) — Step 1: connect Drive and load the agriculture data

import os, shutil

# Connect Google Drive (clear the spot first if it isn't connected)
if not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)
from google.colab import drive
drive.mount('/content/drive')

# Where we'll keep Task 2 files
folder = "/content/drive/MyDrive/Ashesi/ICS554_Prosit1_TaskC"
os.makedirs(folder, exist_ok=True)

# Load the Kenya portion of the farmer questions
from datasets import load_dataset
data = load_dataset("DigiGreen/farmerchat-queries", "kenya", split="train")

print(data)

# Look at the first two rows to see what the text is like
for row in data.select(range(2)):
    print(row)

Mounted at /content/drive


README.md:   0%|          | 0.00/3.01k [00:00<?, ?B/s]

kenya.parquet: reconstructing file:   0%|          |  0.00B / 6.65MB            

kenya.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/6815 [00:00<?, ? examples/s]

Dataset({
    features: ['crop', 'user_country', 'user_geo_level2', 'user_geo_level3', 'query_year_month', 'query', 'response'],
    num_rows: 6815
})
{'crop': 'generic', 'user_country': 'Kenya', 'user_geo_level2': 'Nyeri County', 'user_geo_level3': 'Mukurwe-Ini', 'query_year_month': '2026-02', 'query': 'Where can I find dooper sheep for breeding in Mukurwe-ini near Kabuta?', 'response': "Hello there! It's great to hear that you're interested in breeding Dorper sheep in Mukurwe-ini. Dorper sheep are a fantastic choice due to their adaptability and good meat quality. Let me guide you on where you might find them and how to care for them.\n\nIn Mukurwe-ini, you can start by checking with local farmers who might already be raising Dorper sheep. Networking with other farmers at local markets or agricultural events can be very helpful. Additionally, you might want to visit the nearby agricultural extension office. They often have contacts or can direct you to breeders in the area.\n\nFor ge

In [2]:
# Task 2 (Section C) — Step 2: build the text list and measure it

# Join each question and its answer into one piece of text
texts = [row["query"] + "\n" + row["response"] for row in data]

# Count words across everything
words = [w for t in texts for w in t.split()]

print("Q&A pairs:   ", len(texts))
print("Total words: ", len(words))
print("Different words:", len(set(words)))
print("Average words per pair:", round(len(words) / len(texts)))

Q&A pairs:    6815
Total words:  2295725
Different words: 31777
Average words per pair: 337


In [3]:
# Task 2 (Section C) — check the GPU is available
import torch
print("GPU available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

GPU available: True
Device name: Tesla T4


In [4]:
# Task 2 (Section C) — Step 3: split into train, dev and test
import random

random.seed(42)                       # same split every run
shuffled = texts[:]
random.shuffle(shuffled)

n = len(shuffled)
train_texts = shuffled[: int(n * 0.8)]
dev_texts   = shuffled[int(n * 0.8) : int(n * 0.9)]
test_texts  = shuffled[int(n * 0.9) :]

print("Train:", len(train_texts))
print("Dev:  ", len(dev_texts))
print("Test: ", len(test_texts))

Train: 5452
Dev:   681
Test:  682


In [ ]:
# Task 2 (Section C) — Step 4: measure GPT-2 BEFORE any tuning
import torch, math
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# Download GPT-2 and its tokenizer (the part that chops text into pieces)
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").cuda().eval()

def measure(model, sentences, limit=None):
    total_loss, total_tokens = 0.0, 0
    for t in sentences[:limit]:                      # every piece, unless a limit is given
        ids = tokenizer(t, return_tensors="pt", truncation=True, max_length=512).input_ids.cuda()
        with torch.no_grad():                        # we're only measuring, not training
            loss = model(ids, labels=ids).loss       # average surprise per token
        total_loss += loss.item() * ids.shape[1]
        total_tokens += ids.shape[1]
    return math.exp(total_loss / total_tokens)       # turn surprise into perplexity

before = measure(model, dev_texts)
print("GPT-2 BEFORE tuning — dev perplexity:", round(before, 2))

In [ ]:
# Task 2 (Section C) — Step 5: fine-tune GPT-2 on the farmer text

from torch.utils.data import DataLoader

torch.manual_seed(42)                         # same batches every run
tokenizer.pad_token = tokenizer.eos_token     # GPT-2 has no padding symbol; reuse its end symbol

def to_batch(batch_texts):
    enc = tokenizer(batch_texts, return_tensors="pt", truncation=True,
                    max_length=256, padding="max_length")
    labels = enc.input_ids.masked_fill(enc.attention_mask == 0, -100)   # -100 = don't score the padding
    return enc.input_ids, enc.attention_mask, labels

loader = DataLoader(train_texts, batch_size=4, shuffle=True, collate_fn=to_batch)

train_losses = []                                       # loss at every step, for the learning curve
dev_checks = [(0, measure(model, dev_texts, limit=50))] # (step, dev perplexity), quick check on 50 dev texts

model.train()                                  # switch from measuring to learning
optimiser = torch.optim.AdamW(model.parameters(), lr=5e-5)

for step, (ids, mask, labels) in enumerate(loader, start=1):
    loss = model(ids.cuda(), attention_mask=mask.cuda(), labels=labels.cuda()).loss   # how wrong the model was
    loss.backward()                             # work out how to improve
    optimiser.step()                            # adjust the model
    optimiser.zero_grad()
    train_losses.append(loss.item())
    if step % 100 == 0:                         # every 100 steps, check on text it never trains on
        model.eval()
        dev_checks.append((step, measure(model, dev_texts, limit=50)))
        model.train()
        print("step", step, "loss", round(loss.item(), 3), "dev perplexity", round(dev_checks[-1][1], 2))
    if step == 600:                             # stop early to stay inside the free GPU limit
        break

print("Training done.")

In [ ]:
# Task 2 (Section C) — Step 5b: plot the learning curves

import matplotlib.pyplot as plt

fig_dir = folder + "/figures"                 # figures are saved to Drive for the report
os.makedirs(fig_dir, exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"           # same chart style as Section B
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
})

# Single steps are noisy, so also show the average of the last 25 steps
window = 25
smooth = [sum(train_losses[max(0, i - window + 1): i + 1]) / len(train_losses[max(0, i - window + 1): i + 1])
          for i in range(len(train_losses))]
steps = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(steps, train_losses, color=BLUE, linewidth=1, alpha=0.25, label="each step")
ax1.plot(steps, smooth, color=BLUE, linewidth=2, label=f"average of last {window} steps")
ax1.set_title("Training loss falls as GPT-2 learns the farmer text")
ax1.set_xlabel("training step")
ax1.set_ylabel("loss (cross-entropy)")
ax1.legend(frameon=False)

check_steps, check_ppl = zip(*dev_checks)
ax2.plot(check_steps, check_ppl, color=ORANGE, linewidth=2, marker="o", markersize=7)
for s, p in [dev_checks[0], dev_checks[-1]]:
    ax2.annotate(f"{p:.1f}", (s, p), xytext=(0, 8), textcoords="offset points", ha="center", color=MUTED)
ax2.set_title("Dev perplexity drops on text it never trained on")
ax2.set_xlabel("training step")
ax2.set_ylabel("dev perplexity (50 texts, lower is better)")

plt.tight_layout()
plt.savefig(fig_dir + "/training_curves.png", dpi=150)
plt.show()

In [ ]:
# Task 2 (Section C) — Step 6: measure GPT-2 AFTER tuning

model.eval()                                    # switch back from learning to measuring
after = measure(model, dev_texts)

print("BEFORE tuning:", round(before, 2))
print("AFTER tuning: ", round(after, 2))
print("Improvement:  ", round(before - after, 2))

In [ ]:
# Task 2 (Section C) — Step 7: compare what each model writes

def write(model, prompt, words=40):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()
    out = model.generate(ids, max_new_tokens=words, do_sample=True,
                         top_p=0.9, temperature=0.8,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Load a fresh, untuned GPT-2 (the tuned one was changed in place during training)
base = GPT2LMHeadModel.from_pretrained("gpt2").cuda().eval()

prompts = [
    "What are the best practices for planting beans in",
    "My maize leaves are turning yellow. What should I",
    "How much feed does a dairy cow need",
]

samples = []
for prompt in prompts:
    torch.manual_seed(0); untuned_text = write(base, prompt)    # same random seed for both,
    torch.manual_seed(0); tuned_text = write(model, prompt)     # so only the model differs
    samples.append((prompt, untuned_text, tuned_text))
    print("PROMPT:", prompt)
    print("\nUNTUNED:\n", untuned_text)
    print("\nTUNED:\n", tuned_text)
    print("\n" + "=" * 80 + "\n")

In [ ]:
# Task 2 (Section C) — Step 8: final measurement on the test set

final = measure(model, test_texts)
print("Tuned model — test perplexity:", round(final, 2))

In [ ]:
# Task 2 (Section C) — Step 9: measure the untuned model on test, then save all results

base_test = measure(base, test_texts)     # untuned GPT-2 from Step 7, same test set

lines = [
    "Section C - farmer agriculture domain (Kenya)",
    f"Q&A pairs: {len(texts)}, words: {len(words):,}",
    f"Split: train {len(train_texts)}, dev {len(dev_texts)}, test {len(test_texts)}",
    "Perplexity measured on every dev and test text",
    "",
    f"GPT-2 untuned  - dev perplexity:  {before:.2f}",
    f"GPT-2 tuned    - dev perplexity:  {after:.2f}",
    f"GPT-2 untuned  - test perplexity: {base_test:.2f}",
    f"GPT-2 tuned    - test perplexity: {final:.2f}",
    "",
    "Training: 600 steps, batch size 4, max length 256, learning rate 5e-5, Tesla T4 GPU",
    f"Loss fell from {train_losses[0]:.2f} to {sum(train_losses[-50:]) / 50:.2f} (average of last 50 steps)",
    "Dev perplexity during training (50 texts): "
        + ", ".join(f"step {s}: {p:.2f}" for s, p in dev_checks),
    "",
    "Sample outputs (same prompt and random seed):",
]
for prompt, untuned_text, tuned_text in samples:
    lines += ["", f"PROMPT: {prompt}", f"UNTUNED: {untuned_text}", f"TUNED: {tuned_text}"]

for line in lines:
    print(line)

with open(folder + "/section_c_results.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print("\nSaved to Drive.")

In [ ]:
# Task 2 (Section C) — Step 9b: chart perplexity before and after tuning

import numpy as np

x = np.arange(2)
w = 0.38

fig, ax = plt.subplots(figsize=(7, 4.5))
b1 = ax.bar(x - w/2 - 0.01, [before, base_test], w, color=BLUE, label="GPT-2 untuned")
b2 = ax.bar(x + w/2 + 0.01, [after, final], w, color=ORANGE, label="GPT-2 tuned")
for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.2f", padding=3, color=MUTED, fontsize=9)
ax.set_xticks(x, ["dev", "test"])
ax.set_ylabel("perplexity (lower is better)")
ax.set_title(f"Tuning cuts test perplexity from {base_test:.1f} to {final:.1f}")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.savefig(fig_dir + "/perplexity_before_after.png", dpi=150)
plt.show()

print("Figures saved to", fig_dir)